# RomaSub.AI — Fine-tuning M2M100 for Urdu → Roman-Urdu Transliteration

**Project:** RomaSub.AI (Final Year Project)

**Objective:** Fine-tune `facebook/m2m100_418M` for Urdu → Roman-Urdu transliteration

**Approach:**
1. Download & manually split Roman-Urdu-Parl (6.3M pairs) and Google Dakshina datasets
2. Add custom `__roman_ur__` language token to m2m100 tokenizer
3. Phase 1: Train on RUP (1.5M subset) for 3 epochs
4. Phase 2: Fine-tune on Dakshina for 5 epochs
5. Evaluate with BLEU, Char-BLEU, CHRF, WER, CER, Exact Match, Precision/Recall/F1

**Runtime:** Colab Pro with A100 GPU (~6 compute units/hour)

## Cell 1: Setup & Dependencies

In [ ]:
!pip install -q transformers datasets sentencepiece sacrebleu jiwer accelerate matplotlib seaborn pandas numpy

import torch
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

# Verify GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Create output directories
DRIVE_DIR = '/content/drive/MyDrive/RomaSub_AI'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
RESULTS_DIR = f'{DRIVE_DIR}/results'
FIGURES_DIR = f'{DRIVE_DIR}/figures'
MODEL_DIR = f'{DRIVE_DIR}/final_model'

for d in [CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("Setup complete!")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Cell 2: Download & Load Datasets

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

# ---- 1. Load Roman-Urdu-Parl (RUP) ----
print("Loading Roman-Urdu-Parl dataset...")
rup_raw = load_dataset("Mavkif/Roman-Urdu-Parl-split")

print(f"RUP Train: {len(rup_raw['train']):,} pairs")
print(f"RUP Validation: {len(rup_raw['validation']):,} pairs")
print(f"RUP Test: {len(rup_raw['test']):,} pairs")
print(f"\nSample: {rup_raw['train'][0]}")

# Standardize column names
def rename_rup_columns(example):
    return {
        'urdu': example['Urdu text'],
        'roman_urdu': example['Roman-Urdu text']
    }

rup_all = rup_raw.map(rename_rup_columns, remove_columns=rup_raw['train'].column_names)
print(f"\nStandardized sample: {rup_all['train'][0]}")

In [ ]:
# ---- 2. Download & Load Google Dakshina ----
print("Downloading Dakshina dataset...")
!wget -q https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar
!tar xf dakshina_dataset_v1.0.tar

# Load Urdu romanized sentences
dakshina_path = 'dakshina_dataset_v1.0/ur/romanized/ur.romanized.rejoined.tsv'

# Check if the file exists, try alternate paths
if not os.path.exists(dakshina_path):
    import glob
    matches = glob.glob('dakshina_dataset_v1.0/**/ur.romanized*', recursive=True)
    print(f"Found files: {matches}")
    if matches:
        dakshina_path = matches[0]

# Parse manually to handle lines with extra tabs
rows = []
with open(dakshina_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            # First column is Urdu, second is Roman-Urdu (ignore extra columns)
            rows.append({'urdu': parts[0], 'roman_urdu': parts[1]})

dakshina_df = pd.DataFrame(rows)
print(f"\nDakshina loaded: {len(dakshina_df):,} sentence pairs")
print(f"Sample:\n{dakshina_df.head(3)}")

## Cell 3: Manual Dataset Splitting

Following the paper's strategy (Section 4.1):
- Group by unique Urdu sentences
- Unique sentences (1 Roman-Urdu variant): 1,500 → val, 1,500 → test
- Multi-variant sentences (2-10 variants): 3,000 → val, 3,000 → test
- Remaining → training
- Zero overlap between splits

In [ ]:
# ---- RUP Manual Split ----
# Combine all RUP data for re-splitting
from datasets import concatenate_datasets

rup_combined = concatenate_datasets([rup_all['train'], rup_all['validation'], rup_all['test']])
rup_df = rup_combined.to_pandas()
print(f"Total RUP pairs: {len(rup_df):,}")

# Group by Urdu sentence to find unique vs multi-variant
urdu_groups = rup_df.groupby('urdu')['roman_urdu'].apply(list).reset_index()
urdu_groups['variant_count'] = urdu_groups['roman_urdu'].apply(len)

print(f"\nTotal unique Urdu sentences: {len(urdu_groups):,}")
print(f"Unique (1 variant): {(urdu_groups['variant_count'] == 1).sum():,}")
print(f"Multi-variant (2-10): {((urdu_groups['variant_count'] >= 2) & (urdu_groups['variant_count'] <= 10)).sum():,}")
print(f"Many variants (>10): {(urdu_groups['variant_count'] > 10).sum():,}")

# Separate unique and multi-variant
unique_sentences = urdu_groups[urdu_groups['variant_count'] == 1].copy()
multi_variant = urdu_groups[(urdu_groups['variant_count'] >= 2) & (urdu_groups['variant_count'] <= 10)].copy()

# Shuffle
unique_sentences = unique_sentences.sample(frac=1, random_state=SEED).reset_index(drop=True)
multi_variant = multi_variant.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Split: 1,500 unique for val, 1,500 unique for test
val_unique = unique_sentences.iloc[:1500]
test_unique = unique_sentences.iloc[1500:3000]
remaining_unique = unique_sentences.iloc[3000:]

# Split: 3,000 multi-variant for val, 3,000 for test
val_multi = multi_variant.iloc[:3000]
test_multi = multi_variant.iloc[3000:6000]
remaining_multi = multi_variant.iloc[6000:]

# Collect Urdu sentences for each split
val_urdu_set = set(val_unique['urdu'].tolist() + val_multi['urdu'].tolist())
test_urdu_set = set(test_unique['urdu'].tolist() + test_multi['urdu'].tolist())

print(f"\nValidation Urdu sentences: {len(val_urdu_set):,}")
print(f"Test Urdu sentences: {len(test_urdu_set):,}")
print(f"Overlap between val and test: {len(val_urdu_set & test_urdu_set)}")

# Build final splits from original pairs
rup_val = rup_df[rup_df['urdu'].isin(val_urdu_set)].reset_index(drop=True)
rup_test = rup_df[rup_df['urdu'].isin(test_urdu_set)].reset_index(drop=True)
rup_train = rup_df[~rup_df['urdu'].isin(val_urdu_set | test_urdu_set)].reset_index(drop=True)

print(f"\n--- Final RUP Splits ---")
print(f"Train: {len(rup_train):,} pairs")
print(f"Validation: {len(rup_val):,} pairs")
print(f"Test: {len(rup_test):,} pairs")

# Verify zero overlap
train_urdu = set(rup_train['urdu'])
val_urdu = set(rup_val['urdu'])
test_urdu = set(rup_test['urdu'])

assert len(train_urdu & val_urdu) == 0, "LEAK: Train-Val overlap!"
assert len(train_urdu & test_urdu) == 0, "LEAK: Train-Test overlap!"
assert len(val_urdu & test_urdu) == 0, "LEAK: Val-Test overlap!"
print("\nData integrity check PASSED: Zero overlap between all splits!")

In [ ]:
# ---- Dakshina Split ----
# 500 for test, rest for training (used in fine-tuning phase 2)
dakshina_df = dakshina_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

dakshina_test_df = dakshina_df.iloc[:500]
dakshina_train_df = dakshina_df.iloc[500:]

print(f"Dakshina Train: {len(dakshina_train_df):,} pairs")
print(f"Dakshina Test: {len(dakshina_test_df):,} pairs")

# Convert to HuggingFace datasets
rup_dataset = DatasetDict({
    'train': Dataset.from_pandas(rup_train),
    'validation': Dataset.from_pandas(rup_val),
    'test': Dataset.from_pandas(rup_test)
})

dakshina_dataset = DatasetDict({
    'train': Dataset.from_pandas(dakshina_train_df),
    'test': Dataset.from_pandas(dakshina_test_df)
})

print(f"\nRUP Dataset: {rup_dataset}")
print(f"Dakshina Dataset: {dakshina_dataset}")

In [ ]:
# ---- Create smaller evaluation subsets for faster validation during training ----
# As per the paper: 4,500 sentences for validation/test, subset of 1,500 for quick eval

rup_val_small = rup_dataset['validation'].select(range(min(1500, len(rup_dataset['validation']))))
rup_test_small = rup_dataset['test'].select(range(min(4500, len(rup_dataset['test']))))

print(f"Quick validation subset: {len(rup_val_small)} pairs")
print(f"Test subset: {len(rup_test_small)} pairs")

## Cell 4: Tokenizer Modification & Model Setup

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

MODEL_NAME = 'facebook/m2m100_418M'

print("Loading model and tokenizer...")
tokenizer = M2M100Tokenizer.from_pretrained(MODEL_NAME)
model = M2M100ForConditionalGeneration.from_pretrained(MODEL_NAME)

print(f"Original vocab size: {len(tokenizer)}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# Verify __ur__ exists
print(f"\n'ur' in lang codes: {'ur' in tokenizer.lang_code_to_id}")
print(f"'ur' token ID: {tokenizer.lang_code_to_id.get('ur', 'NOT FOUND')}")

# ---- Add __roman_ur__ token ----
new_lang = 'roman_ur'
new_token = f'__{new_lang}__'

# Get existing special tokens safely
existing_special = tokenizer.special_tokens_map.get('additional_special_tokens', [])
new_special = existing_special + [new_token]

# Add as special token
num_added = tokenizer.add_special_tokens(
    {'additional_special_tokens': new_special}
)
print(f"\nAdded {num_added} new special token(s)")

# Get new token ID
new_token_id = tokenizer.convert_tokens_to_ids(new_token)

# Update ALL internal language mappings
tokenizer.lang_code_to_token[new_lang] = new_token
tokenizer.lang_code_to_id[new_lang] = new_token_id
tokenizer.lang_token_to_id[new_token] = new_token_id  # critical for set_tgt_lang_special_tokens
tokenizer.id_to_lang_code = {v: k for k, v in tokenizer.lang_code_to_id.items()}

# Resize model embeddings
model.resize_token_embeddings(len(tokenizer))

print(f"New vocab size: {len(tokenizer)}")
print(f"'roman_ur' token: {new_token}")
print(f"'roman_ur' token ID: {new_token_id}")

# Verify it works
tokenizer.src_lang = 'ur'
tokenizer.tgt_lang = new_lang  # test target lang setting
test_encode = tokenizer("ٹیسٹ", return_tensors='pt')
print(f"\nTest encoding successful: {test_encode['input_ids'].shape}")

# Get target language ID for generation
FORCED_BOS_TOKEN_ID = tokenizer.lang_code_to_id[new_lang]
print(f"Forced BOS token ID for generation: {FORCED_BOS_TOKEN_ID}")

## Cell 5: Data Preprocessing & Tokenization

In [ ]:
MAX_LENGTH = 128

# Get the target lang token id before mapping (will be used as a constant)
TGT_LANG_TOKEN_ID = tokenizer.lang_code_to_id['roman_ur']
PAD_TOKEN_ID = tokenizer.pad_token_id

# Filter out None/empty values from all datasets before tokenization
def filter_nulls(example):
    return example['urdu'] is not None and example['roman_urdu'] is not None \
        and len(str(example['urdu']).strip()) > 0 and len(str(example['roman_urdu']).strip()) > 0

print("Filtering null/empty values...")
rup_dataset = DatasetDict({
    split: rup_dataset[split].filter(filter_nulls, desc=f"Filtering {split}")
    for split in rup_dataset
})
dakshina_dataset = DatasetDict({
    split: dakshina_dataset[split].filter(filter_nulls, desc=f"Filtering {split}")
    for split in dakshina_dataset
})
rup_val_small = rup_val_small.filter(filter_nulls)

print(f"After filtering — RUP train: {len(rup_dataset['train']):,}, Dakshina train: {len(dakshina_dataset['train']):,}")

def preprocess_function(examples):
    """Tokenize Urdu (source) and Roman-Urdu (target) pairs."""
    # Ensure all inputs are strings
    urdu_texts = [str(t) for t in examples['urdu']]
    roman_texts = [str(t) for t in examples['roman_urdu']]

    # Encode source (Urdu)
    tokenizer.src_lang = 'ur'
    model_inputs = tokenizer(
        urdu_texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )

    # Encode targets (Roman-Urdu) — tokenize as plain text
    # then swap the source language token with target language token
    target_encodings = tokenizer(
        roman_texts,
        max_length=MAX_LENGTH - 1,  # leave room for lang token
        truncation=True,
        padding='max_length'
    )

    # Build labels: [tgt_lang_id, ...tokens..., eos, pad...]
    labels = []
    for input_ids in target_encodings['input_ids']:
        # Replace the source language token (first token) with target language token
        label = [TGT_LANG_TOKEN_ID] + input_ids[1:]
        # Replace pad tokens with -100
        label = [(l if l != PAD_TOKEN_ID else -100) for l in label]
        labels.append(label)

    model_inputs['labels'] = labels
    return model_inputs

# Use all available CPU cores for parallel tokenization
import multiprocessing
NUM_PROC = multiprocessing.cpu_count()
print(f"Using {NUM_PROC} CPU cores for parallel tokenization")

# Tokenize RUP splits
print("Tokenizing RUP dataset...")
rup_tokenized = rup_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=5000,
    num_proc=NUM_PROC,
    remove_columns=['urdu', 'roman_urdu'],
    desc="Tokenizing RUP"
)

# Tokenize validation small subset
rup_val_small_tokenized = rup_val_small.map(
    preprocess_function,
    batched=True,
    batch_size=5000,
    remove_columns=['urdu', 'roman_urdu'],
    desc="Tokenizing RUP val small"
)

# Tokenize Dakshina
print("\nTokenizing Dakshina dataset...")
dakshina_tokenized = dakshina_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=5000,
    remove_columns=['urdu', 'roman_urdu'],
    desc="Tokenizing Dakshina"
)

print(f"\nRUP tokenized: {rup_tokenized}")
print(f"Dakshina tokenized: {dakshina_tokenized}")

## Cell 6: Training Configuration

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import sacrebleu

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Using {'bf16' if use_bf16 else 'fp16'} precision")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding='longest', max_length=MAX_LENGTH
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
    char_preds = [' '.join(list(p)) for p in decoded_preds]
    char_labels = [' '.join(list(l)) for l in decoded_labels]
    char_bleu = sacrebleu.corpus_bleu(char_preds, [char_labels])
    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels])
    return {'bleu': round(bleu.score, 2), 'char_bleu': round(char_bleu.score, 2), 'chrf': round(chrf.score, 2)}

TRAIN_SUBSET_SIZE = 1_500_000
NUM_EPOCHS = 3

full_train = rup_tokenized['train']
if len(full_train) > TRAIN_SUBSET_SIZE:
    train_subset = full_train.shuffle(seed=SEED).select(range(TRAIN_SUBSET_SIZE))
    print(f"Using {TRAIN_SUBSET_SIZE:,} training samples (subset of {len(full_train):,})")
else:
    train_subset = full_train
    print(f"Using full training set: {len(full_train):,}")

# Clear GPU cache before training
torch.cuda.empty_cache()

rup_training_args = Seq2SeqTrainingArguments(
    output_dir=f'{CHECKPOINT_DIR}/rup_phase1',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=1e-5,
    weight_decay=0.02,
    warmup_steps=500,
    bf16=use_bf16,
    fp16=use_fp16,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='char_bleu',
    greater_is_better=True,
    logging_steps=200,
    report_to='none',
    dataloader_num_workers=4,
    seed=SEED,
)

# gradient_checkpointing requires use_cache=False on M2M100
model.config.use_cache = False

steps_per_epoch = TRAIN_SUBSET_SIZE // 256
total_steps = steps_per_epoch * NUM_EPOCHS
print("Training configuration ready!")
print(f"Epochs: {NUM_EPOCHS}, Per-device batch: 32, Accum: 8 -> effective 256")
print(f"Optimizer steps per epoch: ~{steps_per_epoch:,}, Total: ~{total_steps:,}")

## Cell 7: Phase 1 — Train on RUP (1.5M subset, 3 Epochs)

In [ ]:
# Set forced BOS token for generation
model.config.forced_bos_token_id = FORCED_BOS_TOKEN_ID

trainer = Seq2SeqTrainer(
    model=model,
    args=rup_training_args,
    train_dataset=train_subset,  # Using 1.5M subset
    eval_dataset=rup_val_small_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Auto-resume from checkpoint if Colab disconnected mid-training
import glob

phase1_checkpoints = glob.glob(f'{CHECKPOINT_DIR}/rup_phase1/checkpoint-*')
# Sort numerically by step number (not alphabetically!)
phase1_checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
resume_checkpoint = phase1_checkpoints[-1] if phase1_checkpoints else None

if resume_checkpoint:
    print(f"Resuming Phase 1 from checkpoint: {resume_checkpoint}")
else:
    print("Starting Phase 1: Training on RUP (1.5M subset) for 3 epochs...")

print("Checkpoints saved to Google Drive after each epoch.\n")

rup_train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

rup_metrics = rup_train_result.metrics
print(f"\nPhase 1 Training Complete!")
print(f"Training Loss: {rup_metrics.get('train_loss', 'N/A')}")
print(f"Training Runtime: {rup_metrics.get('train_runtime', 0) / 3600:.1f} hours")

phase1_dir = f'{CHECKPOINT_DIR}/rup_phase1_best'
trainer.save_model(phase1_dir)
tokenizer.save_pretrained(phase1_dir)
print(f"\nPhase 1 best model saved to {phase1_dir}")

In [ ]:
# Save training history for plots
rup_log_history = trainer.state.log_history

with open(f'{RESULTS_DIR}/rup_training_log.json', 'w') as f:
    json.dump(rup_log_history, f, indent=2)

print(f"Training log saved ({len(rup_log_history)} entries)")

# Quick peek at epoch-wise eval results
for entry in rup_log_history:
    if 'eval_char_bleu' in entry:
        eval_loss = entry.get('eval_loss', None)
        loss_str = f"{eval_loss:.4f}" if isinstance(eval_loss, (int, float)) else str(eval_loss)
        print(f"Epoch {entry.get('epoch', '?')}: "
              f"Char-BLEU={entry['eval_char_bleu']}, "
              f"BLEU={entry.get('eval_bleu', 'N/A')}, "
              f"CHRF={entry.get('eval_chrf', 'N/A')}, "
              f"Loss={loss_str}")

## Cell 8: Phase 2 — Fine-tune on Dakshina (5 Epochs)

In [ ]:
# Load the best phase 1 model (or use existing if still in memory)
# Uncomment below if restarting from a saved checkpoint:
# model = M2M100ForConditionalGeneration.from_pretrained(phase1_dir)
# tokenizer = M2M100Tokenizer.from_pretrained(phase1_dir)

torch.cuda.empty_cache()

# gradient_checkpointing requires use_cache=False on M2M100
model.config.use_cache = False

dakshina_training_args = Seq2SeqTrainingArguments(
    output_dir=f'{CHECKPOINT_DIR}/dakshina_phase2',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=1e-5,
    weight_decay=0.02,
    warmup_steps=50,
    bf16=use_bf16,
    fp16=use_fp16,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='char_bleu',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    seed=SEED,
)

dakshina_trainer = Seq2SeqTrainer(
    model=model,
    args=dakshina_training_args,
    train_dataset=dakshina_tokenized['train'],
    eval_dataset=dakshina_tokenized['test'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Auto-resume from checkpoint if Colab disconnected mid-training
phase2_checkpoints = glob.glob(f'{CHECKPOINT_DIR}/dakshina_phase2/checkpoint-*')
phase2_checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
resume_checkpoint_p2 = phase2_checkpoints[-1] if phase2_checkpoints else None

if resume_checkpoint_p2:
    print(f"Resuming Phase 2 from checkpoint: {resume_checkpoint_p2}")
else:
    print("Starting Phase 2: Fine-tuning on Dakshina for 5 epochs...")

dakshina_train_result = dakshina_trainer.train(resume_from_checkpoint=resume_checkpoint_p2)

dak_metrics = dakshina_train_result.metrics
print(f"\nPhase 2 Complete!")
print(f"Training Loss: {dak_metrics.get('train_loss', 'N/A')}")

dakshina_trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"\nFinal model saved to {MODEL_DIR}")

dak_log_history = dakshina_trainer.state.log_history
with open(f'{RESULTS_DIR}/dakshina_training_log.json', 'w') as f:
    json.dump(dak_log_history, f, indent=2)

## Cell 9: Comprehensive Evaluation — All Metrics

Evaluating on both RUP test set and Dakshina test set with:
- BLEU (4-gram)
- Char-BLEU (character-level)
- CHRF (Character F-score)
- WER (Word Error Rate)
- CER (Character Error Rate)
- Exact Match Accuracy
- Precision / Recall / F1 (character n-gram level)

In [ ]:
from jiwer import wer as compute_wer, cer as compute_cer

def generate_predictions(model, tokenizer, dataset, batch_size=64):
    """Generate predictions for a dataset."""
    model.eval()
    all_preds = []

    # Get raw text from the un-tokenized dataset
    sources = dataset['urdu']
    references = dataset['roman_urdu']

    tokenizer.src_lang = 'ur'

    for i in range(0, len(sources), batch_size):
        batch_src = sources[i:i + batch_size]

        inputs = tokenizer(
            batch_src,
            return_tensors='pt',
            max_length=MAX_LENGTH,
            truncation=True,
            padding=True
        ).to(model.device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=FORCED_BOS_TOKEN_ID,
                max_length=MAX_LENGTH,
                num_beams=4,  # Beam search for final evaluation
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        all_preds.extend([p.strip() for p in decoded])

        if (i // batch_size) % 10 == 0:
            print(f"  Processed {min(i + batch_size, len(sources))}/{len(sources)}")

    return all_preds, references


def compute_all_metrics(predictions, references):
    """Compute all evaluation metrics."""
    results = {}

    # Filter out empty pairs
    valid_pairs = [(p, r) for p, r in zip(predictions, references) if p and r]
    preds = [p for p, _ in valid_pairs]
    refs = [r for _, r in valid_pairs]

    # 1. BLEU (4-gram, word-level)
    bleu_result = sacrebleu.corpus_bleu(preds, [refs])
    results['bleu'] = round(bleu_result.score, 2)

    # 2. Char-BLEU (character-level)
    char_preds = [' '.join(list(p)) for p in preds]
    char_refs = [' '.join(list(r)) for r in refs]
    char_bleu_result = sacrebleu.corpus_bleu(char_preds, [char_refs])
    results['char_bleu'] = round(char_bleu_result.score, 2)

    # 3. CHRF
    chrf_result = sacrebleu.corpus_chrf(preds, [refs])
    results['chrf'] = round(chrf_result.score, 2)

    # 4. WER (Word Error Rate)
    wer_score = compute_wer(refs, preds)
    results['wer'] = round(wer_score * 100, 2)

    # 5. CER (Character Error Rate)
    cer_score = compute_cer(refs, preds)
    results['cer'] = round(cer_score * 100, 2)

    # 6. Exact Match Accuracy
    exact_matches = sum(1 for p, r in zip(preds, refs) if p == r)
    results['exact_match'] = round(exact_matches / len(preds) * 100, 2)

    # 7. Precision / Recall / F1 (character n-gram level)
    # Using character 4-grams
    total_precision = 0
    total_recall = 0
    total_f1 = 0
    n = 4

    for pred, ref in zip(preds, refs):
        pred_ngrams = Counter([pred[i:i+n] for i in range(len(pred) - n + 1)])
        ref_ngrams = Counter([ref[i:i+n] for i in range(len(ref) - n + 1)])

        if not pred_ngrams or not ref_ngrams:
            continue

        # Overlap
        overlap = sum((pred_ngrams & ref_ngrams).values())
        pred_total = sum(pred_ngrams.values())
        ref_total = sum(ref_ngrams.values())

        p = overlap / pred_total if pred_total > 0 else 0
        r = overlap / ref_total if ref_total > 0 else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

        total_precision += p
        total_recall += r
        total_f1 += f1

    count = len(preds)
    results['char4_precision'] = round(total_precision / count * 100, 2)
    results['char4_recall'] = round(total_recall / count * 100, 2)
    results['char4_f1'] = round(total_f1 / count * 100, 2)

    return results

print("Evaluation functions ready!")

In [ ]:
# ---- Evaluate on RUP Test Set ----
print("=" * 60)
print("Evaluating on RUP Test Set...")
print("=" * 60)

rup_test_data = rup_dataset['test']
rup_preds, rup_refs = generate_predictions(model, tokenizer, rup_test_data)
rup_results = compute_all_metrics(rup_preds, rup_refs)

print("\n--- RUP Test Results ---")
for metric, value in rup_results.items():
    print(f"  {metric:20s}: {value}")

# ---- Evaluate on Dakshina Test Set ----
print("\n" + "=" * 60)
print("Evaluating on Dakshina Test Set...")
print("=" * 60)

dak_test_data = dakshina_dataset['test']
dak_preds, dak_refs = generate_predictions(model, tokenizer, dak_test_data)
dak_results = compute_all_metrics(dak_preds, dak_refs)

print("\n--- Dakshina Test Results ---")
for metric, value in dak_results.items():
    print(f"  {metric:20s}: {value}")

# Save all results
all_results = {
    'rup_test': rup_results,
    'dakshina_test': dak_results
}

with open(f'{RESULTS_DIR}/evaluation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\nResults saved to {RESULTS_DIR}/evaluation_results.json")

In [ ]:
# Save predictions for error analysis
rup_pred_df = pd.DataFrame({
    'urdu': rup_test_data['urdu'],
    'reference': rup_refs,
    'prediction': rup_preds
})
rup_pred_df.to_csv(f'{RESULTS_DIR}/rup_predictions.csv', index=False)

dak_pred_df = pd.DataFrame({
    'urdu': dak_test_data['urdu'],
    'reference': dak_refs,
    'prediction': dak_preds
})
dak_pred_df.to_csv(f'{RESULTS_DIR}/dakshina_predictions.csv', index=False)

print("Predictions saved for error analysis!")

## Cell 10: Confusion & Error Analysis

In [ ]:
def detailed_error_analysis(predictions, references, dataset_name):
    """Perform detailed error analysis."""
    print(f"\n{'='*60}")
    print(f"Error Analysis: {dataset_name}")
    print(f"{'='*60}")

    # 1. Sentence-level error distribution
    cer_scores = []
    wer_scores = []
    lengths = []

    char_substitutions = Counter()
    word_errors = []

    for pred, ref in zip(predictions, references):
        if not pred or not ref:
            continue

        # Per-sentence CER
        try:
            s_cer = compute_cer([ref], [pred])
            s_wer = compute_wer([ref], [pred])
        except Exception:
            continue

        cer_scores.append(s_cer)
        wer_scores.append(s_wer)
        lengths.append(len(ref))

        # Character-level substitution tracking
        for i, (c_pred, c_ref) in enumerate(zip(pred, ref)):
            if c_pred != c_ref:
                char_substitutions[(c_ref, c_pred)] += 1

        # Track worst predictions
        if s_cer > 0.3:  # High error sentences
            word_errors.append({
                'reference': ref,
                'prediction': pred,
                'cer': round(s_cer, 4)
            })

    # 2. Error distribution stats
    cer_arr = np.array(cer_scores)
    print(f"\n--- CER Distribution ---")
    print(f"  Mean CER: {cer_arr.mean():.4f}")
    print(f"  Median CER: {np.median(cer_arr):.4f}")
    print(f"  Std CER: {cer_arr.std():.4f}")
    print(f"  Perfect (CER=0): {(cer_arr == 0).sum()} / {len(cer_arr)} ({(cer_arr == 0).mean()*100:.1f}%)")
    print(f"  Low error (CER<0.1): {(cer_arr < 0.1).sum()} ({(cer_arr < 0.1).mean()*100:.1f}%)")
    print(f"  High error (CER>0.3): {(cer_arr > 0.3).sum()} ({(cer_arr > 0.3).mean()*100:.1f}%)")

    # 3. Most common character substitutions
    print(f"\n--- Top 20 Character Substitutions ---")
    print(f"  {'Reference':>12} → {'Predicted':>12}  Count")
    for (ref_char, pred_char), count in char_substitutions.most_common(20):
        ref_display = repr(ref_char)
        pred_display = repr(pred_char)
        print(f"  {ref_display:>12} → {pred_display:>12}  {count}")

    # 4. Best and worst examples
    print(f"\n--- 5 Best Predictions (Perfect Match) ---")
    perfect = [(p, r) for p, r, c in zip(predictions, references, cer_scores) if c == 0]
    for p, r in perfect[:5]:
        print(f"  REF: {r}")
        print(f"  PRD: {p}")
        print()

    print(f"\n--- 5 Worst Predictions ---")
    worst = sorted(word_errors, key=lambda x: x['cer'], reverse=True)[:5]
    for w in worst:
        print(f"  REF: {w['reference']}")
        print(f"  PRD: {w['prediction']}")
        print(f"  CER: {w['cer']}")
        print()

    return {
        'cer_scores': cer_scores,
        'wer_scores': wer_scores,
        'lengths': lengths,
        'char_substitutions': char_substitutions,
        'word_errors': word_errors
    }

# Run error analysis
rup_errors = detailed_error_analysis(rup_preds, rup_refs, "RUP Test Set")
dak_errors = detailed_error_analysis(dak_preds, dak_refs, "Dakshina Test Set")

## Cell 11: Visualizations for FYP Report

In [ ]:
# Set style for all plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
FIGSIZE = (10, 6)
DPI = 300

# ---- Plot 1: Training & Validation Loss Curves ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phase 1: RUP
with open(f'{RESULTS_DIR}/rup_training_log.json', 'r') as f:
    rup_logs = json.load(f)

train_losses = [(e['step'], e['loss']) for e in rup_logs if 'loss' in e and 'eval_loss' not in e]
eval_entries = [e for e in rup_logs if 'eval_loss' in e]

if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, alpha=0.7, label='Training Loss', color='#2196F3')

if eval_entries:
    eval_epochs = [e['epoch'] for e in eval_entries]
    eval_losses = [e['eval_loss'] for e in eval_entries]
    # Plot eval loss on secondary x-axis mapped to steps
    eval_steps = [e.get('step', 0) for e in eval_entries]
    axes[0].plot(eval_steps, eval_losses, 'o-', label='Validation Loss', color='#FF5722', linewidth=2)

axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Loss')
axes[0].set_title('Phase 1: Training on RUP')
axes[0].legend()

# Phase 2: Dakshina
with open(f'{RESULTS_DIR}/dakshina_training_log.json', 'r') as f:
    dak_logs = json.load(f)

dak_train_losses = [(e['step'], e['loss']) for e in dak_logs if 'loss' in e and 'eval_loss' not in e]
dak_eval_entries = [e for e in dak_logs if 'eval_loss' in e]

if dak_train_losses:
    steps, losses = zip(*dak_train_losses)
    axes[1].plot(steps, losses, alpha=0.7, label='Training Loss', color='#2196F3')

if dak_eval_entries:
    eval_steps = [e.get('step', 0) for e in dak_eval_entries]
    eval_losses = [e['eval_loss'] for e in dak_eval_entries]
    axes[1].plot(eval_steps, eval_losses, 'o-', label='Validation Loss', color='#FF5722', linewidth=2)

axes[1].set_xlabel('Training Steps')
axes[1].set_ylabel('Loss')
axes[1].set_title('Phase 2: Fine-tuning on Dakshina')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/01_training_loss_curves.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: 01_training_loss_curves.png")

In [ ]:
# ---- Plot 2: Epoch-wise Char-BLEU Progress (Phase 1) ----
if eval_entries:
    fig, ax = plt.subplots(figsize=FIGSIZE)

    epochs = [e['epoch'] for e in eval_entries]
    char_bleus = [e.get('eval_char_bleu', 0) for e in eval_entries]
    bleus = [e.get('eval_bleu', 0) for e in eval_entries]
    chrfs = [e.get('eval_chrf', 0) for e in eval_entries]

    ax.plot(epochs, char_bleus, 'o-', label='Char-BLEU', linewidth=2, markersize=8)
    ax.plot(epochs, bleus, 's-', label='BLEU', linewidth=2, markersize=8)
    ax.plot(epochs, chrfs, '^-', label='CHRF', linewidth=2, markersize=8)

    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Phase 1: Metrics per Epoch (RUP Validation)', fontsize=14)
    ax.legend(fontsize=11)
    ax.set_xticks(epochs)

    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/02_epoch_metrics_progress.png', dpi=DPI, bbox_inches='tight')
    plt.show()
    print("Saved: 02_epoch_metrics_progress.png")

In [ ]:
# ---- Plot 3: Metric Comparison Bar Chart (RUP vs Dakshina) ----
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ['bleu', 'char_bleu', 'chrf', 'exact_match', 'char4_precision', 'char4_recall', 'char4_f1']
metric_labels = ['BLEU', 'Char-BLEU', 'CHRF', 'Exact Match\n(%)', 'Char-4gram\nPrecision', 'Char-4gram\nRecall', 'Char-4gram\nF1']

rup_vals = [rup_results.get(m, 0) for m in metrics]
dak_vals = [dak_results.get(m, 0) for m in metrics]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, rup_vals, width, label='RUP Test', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + width/2, dak_vals, width, label='Dakshina Test', color='#FF9800', alpha=0.85)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Urdu → Roman-Urdu: RUP vs Dakshina Test Performance', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03_metric_comparison_rup_vs_dakshina.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: 03_metric_comparison_rup_vs_dakshina.png")

In [ ]:
# ---- Plot 4: Error Rate Metrics (WER & CER) ----
fig, ax = plt.subplots(figsize=(8, 5))

error_metrics = ['wer', 'cer']
error_labels = ['WER (%)', 'CER (%)']

rup_error_vals = [rup_results.get(m, 0) for m in error_metrics]
dak_error_vals = [dak_results.get(m, 0) for m in error_metrics]

x = np.arange(len(error_metrics))
width = 0.3

bars1 = ax.bar(x - width/2, rup_error_vals, width, label='RUP Test', color='#F44336', alpha=0.85)
bars2 = ax.bar(x + width/2, dak_error_vals, width, label='Dakshina Test', color='#E91E63', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11)

ax.set_ylabel('Error Rate (%)', fontsize=12)
ax.set_title('Word Error Rate & Character Error Rate (Lower is Better)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(error_labels, fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/04_error_rates.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: 04_error_rates.png")

In [ ]:
# ---- Plot 5: CER Distribution Histogram ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rup_errors['cer_scores'], bins=50, color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Character Error Rate', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('CER Distribution — RUP Test Set', fontsize=13)
axes[0].axvline(x=np.mean(rup_errors['cer_scores']), color='red', linestyle='--',
               label=f"Mean: {np.mean(rup_errors['cer_scores']):.4f}")
axes[0].legend()

axes[1].hist(dak_errors['cer_scores'], bins=50, color='#FF9800', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Character Error Rate', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('CER Distribution — Dakshina Test Set', fontsize=13)
axes[1].axvline(x=np.mean(dak_errors['cer_scores']), color='red', linestyle='--',
               label=f"Mean: {np.mean(dak_errors['cer_scores']):.4f}")
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_cer_distribution.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: 05_cer_distribution.png")

In [ ]:
# ---- Plot 6: Error Rate vs Sentence Length ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RUP
axes[0].scatter(rup_errors['lengths'], rup_errors['cer_scores'],
               alpha=0.15, s=10, color='#2196F3')
# Add trend line
z = np.polyfit(rup_errors['lengths'], rup_errors['cer_scores'], 1)
p = np.poly1d(z)
x_line = np.linspace(min(rup_errors['lengths']), max(rup_errors['lengths']), 100)
axes[0].plot(x_line, p(x_line), 'r-', linewidth=2, label='Trend')
axes[0].set_xlabel('Reference Length (characters)', fontsize=12)
axes[0].set_ylabel('CER', fontsize=12)
axes[0].set_title('CER vs Sentence Length — RUP', fontsize=13)
axes[0].legend()

# Dakshina
axes[1].scatter(dak_errors['lengths'], dak_errors['cer_scores'],
               alpha=0.3, s=15, color='#FF9800')
z = np.polyfit(dak_errors['lengths'], dak_errors['cer_scores'], 1)
p = np.poly1d(z)
x_line = np.linspace(min(dak_errors['lengths']), max(dak_errors['lengths']), 100)
axes[1].plot(x_line, p(x_line), 'r-', linewidth=2, label='Trend')
axes[1].set_xlabel('Reference Length (characters)', fontsize=12)
axes[1].set_ylabel('CER', fontsize=12)
axes[1].set_title('CER vs Sentence Length — Dakshina', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/06_cer_vs_length.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: 06_cer_vs_length.png")

In [ ]:
# ---- Plot 7: Character Substitution Heatmap ----
def plot_confusion_heatmap(char_subs, title, filename):
    """Plot character substitution confusion matrix."""
    if not char_subs:
        print(f"No character substitutions found for {title} — skipping heatmap.")
        return

    # Get top 15 most confused character pairs
    top_pairs = char_subs.most_common(50)

    # Get unique chars involved
    ref_chars = list(dict.fromkeys([p[0][0] for p in top_pairs]))  # preserve order, deduplicate
    pred_chars = list(dict.fromkeys([p[0][1] for p in top_pairs]))

    # Limit to top 15 for readability
    ref_chars = ref_chars[:15]
    pred_chars = pred_chars[:15]

    if not ref_chars or not pred_chars:
        print(f"Not enough data for heatmap: {title}")
        return

    # Build matrix
    matrix = np.zeros((len(ref_chars), len(pred_chars)))
    for (rc, pc), count in top_pairs:
        if rc in ref_chars and pc in pred_chars:
            matrix[ref_chars.index(rc)][pred_chars.index(pc)] = count

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(matrix, annot=True, fmt='.0f', cmap='YlOrRd',
                xticklabels=[repr(c) for c in pred_chars],
                yticklabels=[repr(c) for c in ref_chars],
                ax=ax)
    ax.set_xlabel('Predicted Character', fontsize=12)
    ax.set_ylabel('Reference Character', fontsize=12)
    ax.set_title(title, fontsize=14)

    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/{filename}', dpi=DPI, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")

plot_confusion_heatmap(
    rup_errors['char_substitutions'],
    'Character Substitution Heatmap — RUP Test',
    '07_char_confusion_rup.png'
)

plot_confusion_heatmap(
    dak_errors['char_substitutions'],
    'Character Substitution Heatmap — Dakshina Test',
    '08_char_confusion_dakshina.png'
)

In [ ]:
# ---- Plot 8: Summary Radar Chart ----
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

categories = ['BLEU', 'Char-BLEU', 'CHRF', 'Exact Match', 'Char4-F1']
radar_metrics = ['bleu', 'char_bleu', 'chrf', 'exact_match', 'char4_f1']

rup_radar = [rup_results.get(m, 0) for m in radar_metrics]
dak_radar = [dak_results.get(m, 0) for m in radar_metrics]

# Close the polygon
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
rup_radar += rup_radar[:1]
dak_radar += dak_radar[:1]
angles += angles[:1]

ax.plot(angles, rup_radar, 'o-', linewidth=2, label='RUP Test', color='#2196F3')
ax.fill(angles, rup_radar, alpha=0.15, color='#2196F3')
ax.plot(angles, dak_radar, 'o-', linewidth=2, label='Dakshina Test', color='#FF9800')
ax.fill(angles, dak_radar, alpha=0.15, color='#FF9800')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 100)
ax.set_title('Model Performance Radar — Urdu → Roman-Urdu', fontsize=14, pad=20)
ax.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/09_radar_chart.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved: 09_radar_chart.png")

## Cell 12: Export Model for FastAPI Integration

In [ ]:
# Save final model in a format ready for FastAPI
EXPORT_DIR = f'{DRIVE_DIR}/final_model_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# Save config for FastAPI integration
integration_config = {
    'model_path': EXPORT_DIR,
    'src_lang': 'ur',
    'tgt_lang': 'roman_ur',
    'forced_bos_token_id': FORCED_BOS_TOKEN_ID,
    'max_length': MAX_LENGTH,
    'num_beams': 4,
    'model_name': 'facebook/m2m100_418M (fine-tuned)',
    'training_details': {
        'phase1': 'RUP 1.5M subset, 3 epochs',
        'phase2': 'Dakshina 5 epochs',
        'batch_size': 256,
        'gradient_accumulation': 1,
        'learning_rate': 1e-5,
        'weight_decay': 0.02,
    },
    'evaluation': all_results
}

with open(f'{EXPORT_DIR}/integration_config.json', 'w') as f:
    json.dump(integration_config, f, indent=2)

print(f"Model exported to: {EXPORT_DIR}")
print(f"\nFiles saved:")
for f in os.listdir(EXPORT_DIR):
    size = os.path.getsize(f'{EXPORT_DIR}/{f}') / 1e6
    print(f"  {f}: {size:.1f} MB")

print(f"\n--- Integration Config ---")
print(json.dumps(integration_config, indent=2))

In [ ]:
# Optional: Push to HuggingFace Hub
# Uncomment and fill in your token to push

# from huggingface_hub import login
# login(token="your_hf_token_here")
#
# HF_REPO = "your-username/romasub-urdu-to-roman-urdu"
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f"Pushed to: https://huggingface.co/{HF_REPO}")

In [ ]:
# Quick test: Transliterate a few sentences
test_sentences = [
    "پاکستان ایک خوبصورت ملک ہے",
    "آج موسم بہت اچھا ہے",
    "میں یونیورسٹی جا رہا ہوں",
    "یہ ہمارا فائنل ایئر پراجیکٹ ہے",
    "مجھے پروگرامنگ بہت پسند ہے",
]

model.eval()
tokenizer.src_lang = 'ur'

print("--- Sample Transliterations ---\n")
for sentence in test_sentences:
    inputs = tokenizer(sentence, return_tensors='pt', max_length=MAX_LENGTH, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=FORCED_BOS_TOKEN_ID,
            max_length=MAX_LENGTH,
            num_beams=4
        )

    output = tokenizer.decode(generated[0], skip_special_tokens=True)
    print(f"  Urdu:       {sentence}")
    print(f"  Roman-Urdu: {output}")
    print()

## Cell 13: Report Summary

In [ ]:
# Generate a formatted summary for the FYP report
print("=" * 70)
print("  ROMASUB.AI — MODEL EVALUATION REPORT")
print("  Urdu → Roman-Urdu Transliteration")
print("  Model: facebook/m2m100_418M (fine-tuned)")
print("=" * 70)

print(f"\n{'Metric':<25} {'RUP Test':>12} {'Dakshina Test':>15}")
print("-" * 55)

display_metrics = [
    ('BLEU (4-gram)', 'bleu'),
    ('Char-BLEU', 'char_bleu'),
    ('CHRF', 'chrf'),
    ('WER (%)', 'wer'),
    ('CER (%)', 'cer'),
    ('Exact Match (%)', 'exact_match'),
    ('Char-4gram Precision', 'char4_precision'),
    ('Char-4gram Recall', 'char4_recall'),
    ('Char-4gram F1', 'char4_f1'),
]

for label, key in display_metrics:
    rup_v = rup_results.get(key, 'N/A')
    dak_v = dak_results.get(key, 'N/A')
    print(f"{label:<25} {rup_v:>12} {dak_v:>15}")

print("-" * 55)

precision_type = "bf16" if use_bf16 else "fp16"

print(f"\n--- Training Details ---")
print(f"  Base Model:         facebook/m2m100_418M (480M parameters)")
print(f"  Phase 1:            RUP dataset (1.5M subset), 3 epochs")
print(f"  Phase 2:            Dakshina dataset, 5 epochs")
print(f"  Batch Size:         256 (no gradient accumulation)")
print(f"  Learning Rate:      1e-5")
print(f"  Max Sequence Length: 128 tokens")
print(f"  Optimizer:          AdamW (weight_decay=0.02)")
print(f"  Precision:          {precision_type}")

print(f"\n--- Dataset Details ---")
print(f"  RUP Train:          {len(rup_dataset['train']):,} pairs")
print(f"  RUP Validation:     {len(rup_dataset['validation']):,} pairs")
print(f"  RUP Test:           {len(rup_dataset['test']):,} pairs")
print(f"  Dakshina Train:     {len(dakshina_dataset['train']):,} pairs")
print(f"  Dakshina Test:      {len(dakshina_dataset['test']):,} pairs")

print(f"\n--- Figures Saved ---")
for fname in sorted(os.listdir(FIGURES_DIR)):
    if fname.endswith('.png'):
        print(f"  {FIGURES_DIR}/{fname}")

print(f"\n--- Results Files ---")
for fname in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {RESULTS_DIR}/{fname}")

print(f"\n{'='*70}")
print(f"  All outputs saved to Google Drive: {DRIVE_DIR}")
print(f"{'='*70}")